In [4]:
import os
import pandas as pd

def audit_dataset(data_dir):
    """
    Scans the directory for CSV/Parquet files, reads a chunk, 
    and reports schema, memory usage, and row counts.
    """
    print(f"--- Data Audit for directory: {data_dir} ---\n")
    
    for file in os.listdir(data_dir):
        if not (file.endswith('.csv') or file.endswith('.parquet')):
            continue
            
        file_path = os.path.join(data_dir, file)
        file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
        
        print(f"📄 File: {file} ({file_size_mb:.2f} MB)")
        
        try:
            if file.endswith('.csv'):
                # Read just 5 rows to get the schema without crashing RAM
                df_sample = pd.read_csv(file_path, nrows=5)
                # Count rows efficiently (Linux/Mac specific, skip if on Windows)
                with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                    row_count = sum(1 for _ in f) - 1 # subtract header
            else:
                df_sample = pd.read_parquet(file_path)
                row_count = len(df_sample)
                df_sample = df_sample.head(5)
                
            print(f"   Rows: {row_count:,}")
            print("   Columns and Data Types:")
            for col in df_sample.columns:
                print(f"    - {col}: {df_sample[col].dtype}")
            print("-" * 50)
            
        except Exception as e:
            print(f"   Error reading {file}: {e}")

# RUN THIS:
audit_dataset(".")

--- Data Audit for directory: . ---

📄 File: lit_deriv.csv (2788.55 MB)
   Rows: 11,381,522
   Columns and Data Types:
    - URL: str
    - accessionNumber: str
    - filingDate: int64
    - filerCik: int64
    - transactionType: str
    - tableRow: int64
    - securityTitle: str
    - securityTitleFn: float64
    - conversionOrExercisePrice: float64
    - conversionOrExercisePriceFn: float64
    - transactionDate: str
    - transactionDateFn: float64
    - deemedExecutionDate: str
    - deemedExecutionDateFn: float64
    - transactionFormType: int64
    - transactionCode: str
    - equitySwapInvolved: int64
    - transactionCodeFn: float64
    - transactionTimeliness: float64
    - transactionTimelinessFn: float64
    - transactionShares: int64
    - transactionSharesFn: float64
    - transactionTotalValue: float64
    - transactionTotalValueFn: float64
    - transactionPricePerShare: float64
    - transactionPricePerShareFn: float64
    - transactionAcquiredDisposedCode: str
    - tr

In [1]:
pip install polars

  Using cached polars-1.43.2-py3-none-any.whl.metadata (11 kB)
  Using cached polars_runtime_32-1.43.2-cp310-abi3-win_amd64.whl.metadata (1.5 kB)
Using cached polars-1.43.2-py3-none-any.whl (847 kB)
Using cached polars_runtime_32-1.43.2-cp310-abi3-win_amd64.whl (52.6 MB)

   ---------------------------------------- 0/2 [polars-runtime-32]
   ---------------------------------------- 0/2 [polars-runtime-32]
   ---------------------------------------- 0/2 [polars-runtime-32]
   ---------------------------------------- 0/2 [polars-runtime-32]
   ---------------------------------------- 0/2 [polars-runtime-32]
   ---------------------------------------- 0/2 [polars-runtime-32]
   ---------------------------------------- 0/2 [polars-runtime-32]
   ---------------------------------------- 0/2 [polars-runtime-32]
   ---------------------------------------- 0/2 [polars-runtime-32]
   -------------------- ------------------- 1/2 [polars]
   -------------------- ------------------- 1/2 [polars]
 

In [1]:
import polars as pl

# Lazily scan the parquet file — nothing is loaded yet
lf = pl.scan_parquet("./processed/ml_ready_transactions.parquet")

result = (
    lf.select([
        pl.col("transactionDate").max().alias("latest_tx"),
        pl.col("filingDate").max().alias("latest_filing"),
    ])
    .collect(engine="streaming")   # use streaming="True" instead of engine=... on older polars versions
)

latest_tx = result["latest_tx"][0]
latest_filing = result["latest_filing"][0]

print(f"Latest Transaction Date in dataset: {latest_tx}")
print(f"Latest Filing Date in dataset:      {latest_filing}")

Latest Transaction Date in dataset: 2050-05-10
Latest Filing Date in dataset:      20260812


## To check number of companies

In [1]:
import polars as pl

PARQUET_PATH = "./processed/ml_ready_transactions.parquet"

print("Calculating total unique companies...")

# A lazy scan to count unique issuerCiks without loading the data into RAM
unique_companies_count = (
    pl.scan_parquet(PARQUET_PATH)
    .select("issuerCik")
    .drop_nulls()
    .select(pl.col("issuerCik").n_unique())
    .collect()
    .item()
)

print(f"Total number of unique companies in the full dataset: {unique_companies_count:,}")

Calculating total unique companies...
Total number of unique companies in the full dataset: 14,195
